# Simulating Conversations with ConvoKitAI
This notebook shows how to run AI agent conversations on ConvoArena (built on [Deliberate Lab](https://github.com/PAIR-code/deliberate-lab)) and get them back as a `convokitai` Corpus, ready for the same analyses as any ConvoKit dataset.

A simulation is described by a **simulation YAML** (the format the mediator toolkit downloads): the agents, an optional mediator and private assistants, the topics, and who talks to whom. `create_simulation` turns each pairing and topic into a ConvoArena experiment, adds the agents to the chat, waits for the conversations to finish, and converts the exports with `export_to_corpus`.

By default everything runs **locally**: a ConvoArena backend runs in a Docker container on your machine, so you don't need a Firebase project. The last section shows how to run the same simulation on **your own ConvoArena deployment** instead.

## Setup
From the repository root, install ConvoKit (which includes `convokitai`) and the simulation package:

```
python -m pip install -e . -e ./convokit/convokitai-simulation
```

To run locally you also need:

- **Docker**, running. The first run pulls the backend image (`ghcr.io/cornellnlp/convoarena-backend:v0.3`). On Google Colab, where Docker isn't available, the package falls back to `udocker` automatically, and startup is slower.
- **A Gemini API key** for the agents' model calls. Without it the agents join the chat but never send a message.

In [ ]:
import os

import yaml
from convokitai import Corpus
from create_simulation import FirebaseBackend, create_simulation, load_simulation_config
from simulation import simulate

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")  # or paste your key here

## 1. The Simulation YAML
The top-level fields set up the conversations:

| Field | Meaning |
|---|---|
| `description` | Text shown with the chat. |
| `blocks` | Text blocks shown with the chat. The first block fills the `{topic_name}` / `{topic_statement}` placeholders in prompts. A block with several `descriptions` runs one conversation per description. |
| `max_utterance`, `max_time` | When the chat ends: after this many messages, or this many minutes. |
| `pairings` | Who is in each conversation. A member is `mediator:<key>` or an agent `<key>`, optionally with a private `assistant`. |
| `definitions` | The `agents`, `mediators` and `assistants` the pairings refer to: persona, model, generation settings and prompt. |

**Number of conversations:** pairings × every combination of block descriptions. The example below has one pairing and two topics, so it runs **two conversations**. Each has a mediator and two agents with different characters, and one agent has a private assistant.

Prompts are lists of items. `TEXT` items are sent as written, after the placeholders are filled. `CHARACTER_CONTEXT` inserts the agent's `persona.character`. `CONTEXT` inserts the chat so far (`current`), the stages before it (`before`), or both (`all`). `PARTICIPANT_INFO` and `PARTICIPANT_CHAT_INPUT` give an assistant the participant it helps and the message they're drafting.

In [8]:
SIM_YAML = """
description: Two people discuss a question with the help of a mediator.
blocks:
  - name: Debate topic
    descriptions:
      - Should cities ban cars from their downtown areas?
      - Should homework be banned in elementary school?
max_utterance: 10
max_time: 3
pairings:
  - id: mediated-debate
    members:
      - participant: mediator:mediator
        assistant: null
      - participant: skeptic
        assistant: tone-checker
      - participant: enthusiast
        assistant: null
definitions:
  agents:
    skeptic:
      name: skeptic
      content:
        persona:
          id: skeptic
          name: Skeptic
          avatar: 🦉
          pronouns: they/them
          character: cautious retiree who distrusts big changes and asks for evidence
        model: {apiType: GEMINI, modelName: gemini-3-flash-preview}
        generation: {temperature: 0.7}
        chatSettings:
          promptMap:
            message:
              order: 1
              prompt:
                - type: TEXT
                  text: "You are a "
                - type: CHARACTER_CONTEXT
                - type: TEXT
                  text: " discussing this question with one other person: {topic_statement}"
                - type: CONTEXT
                  context: current
                - type: TEXT
                  text: Reply to the conversation in 1-2 sentences.
    enthusiast:
      name: enthusiast
      content:
        persona:
          id: enthusiast
          name: Enthusiast
          avatar: 🦊
          pronouns: they/them
          character: optimistic college student who loves bold new ideas
        model: {apiType: GEMINI, modelName: gemini-3-flash-preview}
        generation: {temperature: 0.7}
        chatSettings:
          promptMap:
            message:
              order: 1
              prompt:
                - type: TEXT
                  text: "You are an "
                - type: CHARACTER_CONTEXT
                - type: TEXT
                  text: " discussing this question with one other person: {topic_statement}"
                - type: CONTEXT
                  context: current
                - type: TEXT
                  text: Reply to the conversation in 1-2 sentences.
  mediators:
    mediator:
      name: mediator
      content:
        persona: {id: mediator, name: Mediator, avatar: 🤝}
        model: {apiType: GEMINI, modelName: gemini-3-flash-preview}
        generation: {temperature: 0.7}
        chat_settings: {can_self_trigger_calls: true}
        min_participant_messages_before_responding: 2
        structured_output:
          message_field: response
          explanation_field: explanation
          schema:
            explanation: {type: STRING, description: 1-2 sentences explaining your message or silence.}
            response: {type: STRING, description: Your chat message.}
        prompt:
          - id: 0
            type: TEXT
            text: "You are a mediator in a live conversation about: {topic_statement}"
          - id: 1
            type: CONTEXT
            context: current
          - id: 2
            type: TEXT
            text: Help the participants understand each other's reasons. Keep your message to 1-2 sentences.
  assistants:
    tone-checker:
      name: tone checker
      content:
        persona: {id: tone-checker, name: Tone Checker, avatar: 🤖, min_call_interval_ms: 8000}
        model: {apiType: GEMINI, modelName: gemini-3-flash-preview}
        generation: {temperature: 0.7}
        structured_output:
          message_field: response
          schema:
            response: {type: STRING, description: Your private note to the participant.}
        prompt:
          - id: 0
            type: TEXT
            text: "You privately help this participant:"
          - id: 1
            type: PARTICIPANT_INFO
          - id: 2
            type: TEXT
            text: "They are drafting this message:"
          - id: 3
            type: PARTICIPANT_CHAT_INPUT
          - id: 4
            type: CONTEXT
            context: current
          - id: 5
            type: TEXT
            text: In one sentence, say whether the draft is likely to raise or lower tension.
"""

`sim_yaml` can also be a path to a YAML file (for example one downloaded from the toolkit) or an already-parsed dict.

### Preview the conversations
`load_simulation_config` parses and checks the YAML without a backend, so you can confirm what will run before starting anything.

In [10]:
config = load_simulation_config(SIM_YAML)
n_topics = 1
for block in config["blocks"]:
    n_topics *= len(block.get("descriptions") or [block.get("description", "")])
print(f"{len(config['pairings'])} pairing(s) x {n_topics} topic combination(s) = "
      f"{len(config['pairings']) * n_topics} conversation(s)")
for pairing in config["pairings"]:
    print(f"\n{pairing['id']}:")
    for member in pairing["members"]:
        print(f"  {member['participant']}" + (f" (assistant: {member['assistant']})" if member.get("assistant") else ""))

1 pairing(s) x 2 topic combination(s) = 2 conversation(s)

mediated-debate:
  mediator:mediator
  skeptic (assistant: tone-checker)
  enthusiast


## 2. Run the Simulation Locally
`simulate` starts the backend container and the ConvoArena emulators, saves your Gemini key in the backend, runs every conversation, and stops the backend when it's done.

- **Timing:** the first run pulls the image and can take a few minutes to start. After that, each conversation starts one at a time and then they all run together. It waits up to `max_time` plus 5 minutes (change this with `wait_timeout`, in seconds).
- **Unfinished conversations:** a conversation that hasn't finished by then is kept as it stands, with `convo.meta["completed"] = False`, and the log says where it got stuck (for example a failed model call).

In [12]:
corpus = simulate(gemini_api_key=GEMINI_API_KEY, sim_yaml=SIM_YAML)
corpus.print_summary_stats()

created experiment exp-74d8748d234248fc, cohort muurdkbp-ixyxil (pairing mediated-debate: Should cities ban cars from their downtown areas?)
  skeptic joined
  enthusiast joined
  waiting up to 120s for the conversation to start...
  conversation started
created experiment exp-b57a7fad7bdd4299, cohort muure1ia-n9g5z5 (pairing mediated-debate: Should homework be banned in elementary school?)
  skeptic joined
  enthusiast joined
  waiting up to 120s for the conversation to start...
  conversation started
2 conversation(s) running, waiting for them to finish...


experiments completed: 0/2 |          | waited 00:00

Number of Speakers: 6
Number of Utterances: 16
Number of Conversations: 2
Number of AI Speakers: 6
Number of PrivateAssistants: 2
Number of Supports: 19


To run several simulations without restarting the backend each time, start the container once and pass the backend to `create_simulation` directly:

```python
from dl_container import BackendContainer, FIRESTORE_PORT
from local_backend import LocalBackend
from create_simulation import _seed_gemini_api_key

container = BackendContainer().start()
backend = LocalBackend(".", reuse_running=True, firestore_port=FIRESTORE_PORT, startup_timeout=600).__enter__()
_seed_gemini_api_key(backend, GEMINI_API_KEY)

corpus_a = create_simulation(backend, SIM_YAML)
corpus_b = create_simulation(backend, "another_simulation.yaml")

backend.__exit__(None, None, None)
container.stop()
```

## 3. The Resulting Corpus
Each conversation is one pairing and topic combination. Its metadata records where it came from:

- `pairing_id`: the pairing it ran.
- `completed`: whether it finished before the timeout.
- `blocks`: the topic text shown with the chat.
- `experiment_id`, `experiment_name`, `cohort_name`: the ConvoArena experiment and cohort it ran as.

In [13]:
for convo in corpus.iter_conversations():
    print(convo.id, {k: convo.meta.get(k) for k in ["pairing_id", "completed", "blocks"]})

muurdkbp-ixyxil {'pairing_id': 'mediated-debate', 'completed': True, 'blocks': [{'name': 'Debate topic', 'description': 'Should cities ban cars from their downtown areas?'}]}
muure1ia-n9g5z5 {'pairing_id': 'mediated-debate', 'completed': True, 'blocks': [{'name': 'Debate topic', 'description': 'Should homework be banned in elementary school?'}]}


Speakers are AI agents (`is_ai=True`). The mediator's `ai_meta["role"]` is `"public assistant"`, and the agents' is `"participant"`. `convo.alias` maps speaker IDs to the persona names shown in the chat. `ai_meta["config"]` holds the model, the prompt text and, under `deliberate_lab`, the full ConvoArena template, so the agent can be recreated.

In [14]:
convo = next(corpus.iter_conversations())
for speaker in convo.iter_speakers():
    config = speaker.ai_meta.get("config", {})
    print(f"{convo.alias.get(speaker.id, speaker.id):12s} is_ai={speaker.is_ai}  role={speaker.ai_meta.get('role')}  "
          f"model={config.get('provider')}/{config.get('model')}")

Goose        is_ai=True  role=participant  model=gemini/gemini-3-flash-preview
Mediator     is_ai=True  role=public assistant  model=gemini/gemini-3-flash-preview
Bear         is_ai=True  role=participant  model=gemini/gemini-3-flash-preview


In [15]:
# The conversation, with each private assistant note shown under the message it replies to
print(convo.get_transcript(supports=True))

Mediator: Welcome Bear and Goose! To get started, please share your thoughts on whether cities should ban cars from their downtown areas and the main reasons behind your views.
    PrivateAssistant[tone-checker] -> Bear:
        tone-checker: Since you haven't typed anything yet, I cannot determine whether your message will raise or lower tension.
    PrivateAssistant[tone-checker] -> Bear:
        draft: (04:38) Bear: I’m quite skeptical of such a drastic change and would need to see hard evidence that it won't destroy local businesses or leave seniors like me without a way to get around. Big shifts often have unintended consequences, so I'd like to see proven results before we even consider a ban.
        tone-checker: Your draft is likely to lower tension because it communicates your skepticism through specific, valid concerns and a request for evidence rather than using inflammatory or dismissive language.
    PrivateAssistant[tone-checker] -> Bear:
        tone-checker: Since the 

Messages from an agent whose structured output has an `explanation_field` (here, the mediator) carry its stated reason for the message in `ai_meta["explanation"]`. Private assistant notes are **Supports**: they're attached to the utterance they reply to (`utt.supports`) and grouped by assistant in `convo.ai_meta["private_assistants"]`. Each support keeps the participant's draft at the time (`draft`).

In [16]:
for utt in convo.get_chronological_utterance_list():
    if utt.ai_meta.get("explanation"):
        print(f"{convo.alias[utt.speaker.id]}: {utt.text}\n  why: {utt.ai_meta['explanation']}\n")

Mediator: Welcome Bear and Goose! To get started, please share your thoughts on whether cities should ban cars from their downtown areas and the main reasons behind your views.
  why: I am introducing the topic and inviting both participants to state their initial positions and reasoning.

Bear: I’m quite skeptical of such a drastic change and would need to see hard evidence that it won't destroy local businesses or leave seniors like me without a way to get around. Big shifts often have unintended consequences, so I'd like to see proven results before we even consider a ban.
  why: I kept the draft unchanged because the feedback confirmed that my approach of using specific concerns and requesting evidence effectively lowers tension while maintaining my skeptical stance.

Mediator: Bear is prioritizing the stability of local businesses and mobility for seniors, while Goose suggests that technology and pilot programs could address those specific concerns while revitalizing the area. By 

In [17]:
for assistant in convo.ai_meta["private_assistants"]:
    print(f"{assistant.id} helps {[convo.alias.get(s, s) for s in assistant.speakers]}")
    for support in assistant.supports[:5]:
        print(f"  draft: {support.draft!r}\n  note:  {support.text}")

tone-checker helps ['Bear']
  draft: ''
  note:  Since you haven't typed anything yet, I cannot determine whether your message will raise or lower tension.
  draft: "(04:38) Bear: I’m quite skeptical of such a drastic change and would need to see hard evidence that it won't destroy local businesses or leave seniors like me without a way to get around. Big shifts often have unintended consequences, so I'd like to see proven results before we even consider a ban."
  note:  Your draft is likely to lower tension because it communicates your skepticism through specific, valid concerns and a request for evidence rather than using inflammatory or dismissive language.
  draft: ''
  note:  Since the participant has not yet entered a draft, it is not possible to evaluate its impact on tension.
  draft: "I'm curious to hear Goose's perspective, provided it's backed by real-world data rather than just optimism. We need to know exactly how emergency vehicles and people with mobility issues will be 

In [18]:
# The parsed simulation YAML is stored with the corpus
corpus.ai_meta["simulation_config"]["pairings"]

[{'id': 'mediated-debate',
  'members': [{'participant': 'mediator:mediator', 'assistant': None},
   {'participant': 'skeptic', 'assistant': 'tone-checker'},
   {'participant': 'enthusiast', 'assistant': None}]}]

## 4. Save the Corpus
The corpus saves and loads like any ConvoKit corpus. Because the mediator is marked as `"public assistant"` and speaker names are in `convo.alias`, it works directly with `convokitai_evaluation` (CIG, ClaimTracker and MediatorRedirection) with no extra setup.

In [19]:
corpus.dump("simulation_demo_corpus", base_path=".")
reloaded = Corpus.load("simulation_demo_corpus")
reloaded.print_summary_stats()

Number of Speakers: 6
Number of Utterances: 16
Number of Conversations: 2
Number of AI Speakers: 6
Number of PrivateAssistants: 2
Number of Supports: 19


## 5. Running on Your Own ConvoArena Deployment
To run on a ConvoArena instance you've deployed to Firebase:

1. In your deployment's web UI, go to **Settings → API Keys** and create a ConvoArena API key.
2. In the same **Settings** page, save your Gemini API key. Agents use the key saved by the account that owns the ConvoArena API key, so you don't pass `gemini_api_key` here.
3. Note your Firebase **project ID** and the **region** your Cloud Functions run in (default `us-central1`).

The experiments are created in your deployment, tagged `convokitai-simulation`, so you can also watch the conversations live in the web UI.

In [20]:
FIREBASE_PROJECT_ID = ""  # e.g. "my-convoarena"
DL_API_KEY = os.environ.get("DL_API_KEY", "")  # starts with "dlb_live_"

if FIREBASE_PROJECT_ID:
    firebase_corpus = simulate(sim_yaml=SIM_YAML, project_id=FIREBASE_PROJECT_ID, dl_api_key=DL_API_KEY)
    firebase_corpus.print_summary_stats()

If your functions run outside `us-central1`, build the backend yourself and call `create_simulation` with it:

In [ ]:
if FIREBASE_PROJECT_ID:
    backend = FirebaseBackend(project_id=FIREBASE_PROJECT_ID, api_key=DL_API_KEY, region="us-central1")
    print(backend.client().health_check())  # check the URL and key before running
    firebase_corpus = create_simulation(backend, SIM_YAML, wait_timeout=600)